# 09 · Keyword Search (BM25)

**Idea:** dense vectors match *meaning*, but they blur exact tokens — acronyms, error codes, function names. **BM25** is a sparse, term-frequency ranker that scores a passage by how often the query words appear (dampened by document length and term rarity). It nails the exact matches embeddings miss.

> Needs `pip install rank-bm25`. The dense-comparison cell needs Ollama running.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [ ]:
from rag_lab import load_markdown, BM25Store

corpus = load_markdown()
bm25 = BM25Store()
bm25.add(corpus)
print('indexed', len(bm25), 'chunks (no embeddings needed)')

BM25 needs no model — just tokenized text. Search for an **exact acronym** that appears literally in the corpus:

In [ ]:
query = 'what does RRF stand for?'
for h in bm25.search(query, k=3):
    print(f'{h.score:5.2f}  {h.chunk.source} :: {h.chunk.meta.get("heading")}')

Now compare against **dense** retrieval on the same query. BM25 rewards the literal token `RRF`; the embedding model reasons about meaning. Run both side by side (this cell embeds the corpus with Ollama).

In [ ]:
from rag_lab import NumpyStore

dense = NumpyStore(provider='ollama')
dense.add(corpus)

def show(label, hits):
    print(label)
    for h in hits:
        print(f'  {h.score:5.2f}  {h.chunk.source} :: {h.chunk.meta.get("heading")}')

show('BM25 (sparse):', bm25.search(query, k=3))
show('Dense (vectors):', dense.search(query, k=3))

Try the reverse: a **paraphrase** with no shared keywords. Here dense usually wins because BM25 has no matching terms to score.

In [ ]:
paraphrase = 'how do I keep passages topically coherent?'
show('BM25 (sparse):', bm25.search(paraphrase, k=3))
show('Dense (vectors):', dense.search(paraphrase, k=3))

**Takeaway:** BM25 and dense retrieval fail in *opposite* ways — exact tokens vs. paraphrases. That complementarity is exactly why the next notebook fuses them.

Next → `10_hybrid_search.ipynb`